In [ ]:
# Example 1: Basic App with a single command, invoked programmatically.
# cyclopts v4: App.__call__ applies a result_action (default sys.exit-style) —
# request the raw return value for programmatic use.
from cyclopts import App

app = App(name="demo", help="A small demo app")

@app.command
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"

result = app(["greet", "world"], exit_on_error=False, result_action="return_value")
print(repr(result))
assert result == "Hello, world!"


In [ ]:
# Example 2: Typed parameters - int, float, bool flags are parsed from strings.
from cyclopts import App

app = App(name="calc")

@app.command
def add(a: int, b: int, *, scale: float = 1.0, negate: bool = False) -> float:
    value = (a + b) * scale
    return -value if negate else value

r1 = app(["add", "3", "4"], exit_on_error=False, result_action="return_value")
r2 = app(["add", "3", "4", "--scale", "2.5"], exit_on_error=False, result_action="return_value")
r3 = app(["add", "3", "4", "--negate"], exit_on_error=False, result_action="return_value")
print(r1, r2, r3)
assert r1 == 7.0
assert r2 == 17.5
assert r3 == -7.0


In [ ]:
# Example 3: Parameter annotation with custom help / name override.
from typing import Annotated
from cyclopts import App, Parameter

app = App(name="net")

@app.command
def connect(
    host: Annotated[str, Parameter(help="Target hostname")],
    port: Annotated[int, Parameter(name=["--port", "-p"], help="TCP port")] = 80,
) -> tuple[str, int]:
    return (host, port)

r1 = app(["connect", "example.com"], exit_on_error=False, result_action="return_value")
r2 = app(["connect", "example.com", "-p", "8080"], exit_on_error=False, result_action="return_value")
print(r1, r2)
assert r1 == ("example.com", 80)
assert r2 == ("example.com", 8080)


In [ ]:
# Example 4: Sub-apps (nested commands).
from cyclopts import App

app = App(name="tool")
db = App(name="db", help="Database commands")
app.command(db)

@db.command
def migrate(version: str = "latest") -> str:
    return f"migrated to {version}"

@db.command
def seed(count: int = 10) -> str:
    return f"seeded {count} rows"

r1 = app(["db", "migrate"], exit_on_error=False, result_action="return_value")
r2 = app(["db", "seed", "--count", "42"], exit_on_error=False, result_action="return_value")
print(r1, r2)
assert r1 == "migrated to latest"
assert r2 == "seeded 42 rows"


In [ ]:
# Example 5: Validation error surface + introspection of registered commands.
from cyclopts import App
from cyclopts.exceptions import CycloptsError

app = App(name="validate")

@app.command
def takes_int(n: int) -> int:
    return n * 2

# Happy path.
ok = app(["takes-int", "21"], exit_on_error=False, result_action="return_value")
assert ok == 42

# Bad input -> raises a CycloptsError subclass (not SystemExit, because exit_on_error=False).
raised = False
try:
    app(["takes-int", "not-a-number"], exit_on_error=False, result_action="return_value")
except CycloptsError as e:
    raised = True
    print(f"Got expected error: {type(e).__name__}")
except Exception as e:
    raised = True
    print(f"Got error: {type(e).__name__}: {e}")

registered = sorted(app._commands.keys()) if hasattr(app, "_commands") else []
print(f"Registered commands (best-effort): {registered}")
assert raised, "Expected a parse/validation error on bad int input"
